# DIAMOND hits filtering

Filters the DIAMOND blastp results produced by `run_diamond_annotation.sh`.

For each database (DB1-DB6), every `results_<DB>/*.outfmt6` file is read, two extra columns are computed and the hits are filtered. The result is saved next to the input as `<name>_filtered.tsv` (with header).

**Computed columns**
- `length_ratio` = qlen / slen
- `scovhsp` = alignment length / slen × 100 (subject coverage; DIAMOND only reports query coverage, `qcovhsp`)

**Filters (all must be met)**

| Metric | Cut-off |
|---|---|
| bitscore | > 50 |
| e-value | < 1e-4 |
| identity (`pident`) | > 40 % |
| query coverage (`qcovhsp`) | > 40 % |
| subject coverage (`scovhsp`) | > 40 % |

In [ ]:
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------
DBS = ["DB1", "DB2", "DB3", "DB4", "DB5", "DB6"]

COLUMNS = [
    "qseqid", "sseqid", "qlen", "slen", "length",
    "evalue", "bitscore", "qcovhsp", "pident", "ppos",
]

MIN_BITSCORE = 50
MAX_EVALUE = 1e-4
MIN_PIDENT = 40
MIN_QCOV = 40
MIN_SCOV = 40

In [ ]:
for db in DBS:
    input_dir = Path(f"results_{db}")

    for file in input_dir.glob("*.outfmt6"):
        print(f"Processing {file.name}...")

        # Read DIAMOND tabular output (no header)
        df = pd.read_csv(file, sep="\t", header=None, low_memory=False)
        df.columns = COLUMNS

        # Length ratio and subject coverage
        df["length_ratio"] = df["qlen"] / df["slen"]
        df["scovhsp"] = (df["length"] / df["slen"]) * 100

        # Apply filters
        filtered_df = df[
            (df["bitscore"] > MIN_BITSCORE) &
            (df["evalue"] < MAX_EVALUE) &
            (df["pident"] > MIN_PIDENT) &
            (df["qcovhsp"] > MIN_QCOV) &
            (df["scovhsp"] > MIN_SCOV)
        ].copy()

        # Save next to the input file
        output_file = file.with_name(file.stem + "_filtered.tsv")
        filtered_df.to_csv(output_file, sep="\t", index=False)

    print(f"{db} done.\n")

print("All done!")